https://confident-reflection-production-d304.up.railway.app/api/admin/extract-all?token=CHOCOLATCOGNITIF


In [27]:
from pathlib import Path
import json
import pandas as pd

rows = []

for file in Path("archives").rglob("*.json"):
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for trial in data["resultats"]:
        if trial.get("task") != "target":
            continue

        participant_id = trial["participant_id"]
        rt = trial["rt"]
        if not rt:
            continue

        valence = trial["valence"]
        attr = trial["baseline_attrac"]
        response = trial["response_meaning"]

        # Congruence
        if (valence == "positif" and attr == "Attractif") or \
        (valence == "negatif" and attr == "Non attractif"):
            congruence = "congruent"
        else:
            congruence = "incongruent"
        if valence=='neutre':
            congruence='neutre'

        # Résultat
        if attr == "Attractif" and response == "Attractif":
            result = 1
        elif attr == "Unattractif" and response == "Non attractif":
            result = 1
        else:
            result = 0

        rows.append({
            "id": participant_id,
            "valence_mot":valence,
            "actracttiveness_face":attr, 
            "congruence": congruence,
            "correct": result,
            "rt": rt/1000
        })

# Transformer en DataFrame
df = pd.DataFrame(rows)

# Export en CSV
df.to_csv("resultats.csv", index=False, encoding="utf-8")

print("Fichier CSV généré")

Fichier CSV généré


### Condition controle

Visage précédé d'un mot neutre. En théorie, 100% de précision:

In [41]:
# We take all the rows where the valence of the word is neutral = control condition
control = df[df['congruence'] == 'neutre']
# Pourcentage of correct answer per id
control_pourcentage = control.groupby('id')['correct'].mean()
# Get the index of the id that got a pourcentage of correct answer below 70%
id_outliers = control_pourcentage[control_pourcentage < 0.70].index
# Construct a new datafram without outliers
data_without_outliers = control[~control['id'].isin(id_outliers)]
# Export en CSV
data_without_outliers.to_csv("data_without_outliers.csv", index=False, encoding="utf-8")

In [ ]:
#To verify
data_without_outliers.groupby('id')['correct'].mean().reset_index(name='%')


,id,%
0,1v24kyhs,0.894737
1,2jp4ppt2,0.850000
2,3ofn5wde,0.950000
3,4hgpolkf,0.722222
4,6ipwu4mk,0.750000
5,7re7u25l,0.750000
6,7rngwvq4,0.900000
7,bf915bwb,0.700000
8,cc6c243c,0.700000
9,d5kgyqu9,1.000000


### Condition positive

In [7]:
positive_csv=df[df['valence_mot']=='positif']
positive_csv.to_csv("resultats_condition_positive.csv", index=False, encoding="utf-8")
positive_csv

,id,valence_mot,actracttiveness_face,congruence,correct,rt
0,9at0mps0,positif,Attractif,congruent,0,0.223
1,9at0mps0,positif,Unattractif,incongruent,0,0.295
4,9at0mps0,positif,Unattractif,incongruent,0,1.067
10,9at0mps0,positif,Unattractif,incongruent,0,0.455
12,9at0mps0,positif,Unattractif,incongruent,0,0.407
...,...,...,...,...,...,...
5666,6ipwu4mk,positif,Attractif,congruent,0,0.623
5667,6ipwu4mk,positif,Unattractif,incongruent,1,0.500
5670,6ipwu4mk,positif,Attractif,congruent,0,0.733
5671,6ipwu4mk,positif,Unattractif,incongruent,1,0.478


In [8]:
result_p = (
    positive_csv.groupby('id')['correct']
    .mean()
    .reset_index(name='pourcentage_correct')
)

result_p['pourcentage_correct'] *= 100
result_p['pourcentage_correct'].mean()

np.float64(75.50265625024322)

### Condition negative


In [9]:
negative_csv=df[df['valence_mot']=='negatif']
negative_csv.to_csv("resultats_condition_negatif.csv", index=False, encoding="utf-8")
negative_csv

,id,valence_mot,actracttiveness_face,congruence,correct,rt
2,9at0mps0,negatif,Unattractif,incongruent,1,0.109
6,9at0mps0,negatif,Attractif,incongruent,1,0.222
9,9at0mps0,negatif,Attractif,incongruent,1,0.518
17,9at0mps0,negatif,Unattractif,incongruent,0,0.195
21,9at0mps0,negatif,Attractif,incongruent,1,0.502
...,...,...,...,...,...,...
5668,6ipwu4mk,negatif,Unattractif,incongruent,1,0.640
5669,6ipwu4mk,negatif,Unattractif,incongruent,1,0.629
5674,6ipwu4mk,negatif,Attractif,incongruent,0,0.879
5675,6ipwu4mk,negatif,Attractif,incongruent,1,0.546


In [10]:
result_n = (
    negative_csv.groupby('id')['correct']
    .mean()
    .reset_index(name='pourcentage_correct')
)

result_n['pourcentage_correct'] *= 100
result_n['pourcentage_correct'].mean()

np.float64(77.31110985846007)